# Research reliability audit / 2026-10-22

Recorded inputs only. No live database writes, network, orders or notifications. See REPORT.md and PROTOCOL.md for scope and limitations. The numeric results below were executed when this notebook was saved. Re-running uses the preserved 310-row input snapshot and exchange calendar.

Order: sample independence, result lifetime, source consistency.

In [1]:
import csv, json, sys
from pathlib import Path
root = Path.cwd().resolve()
while not (root / 'scripts/research_reliability.py').exists():
    if root == root.parent:
        raise RuntimeError('Open this notebook from inside the stockAlarm workspace')
    root = root.parent
sys.path.insert(0, str(root))
from scripts.research_reliability import sample_audit
folder = root / 'reports/revalidation/2026-10-22-research-reliability'
recorded = json.loads((folder / 'analysis.json').read_text(encoding='utf-8'))
rows = json.loads((folder / 'sample_rows.json').read_text(encoding='utf-8'))
calendar = json.loads((folder / 'calendar.json').read_text(encoding='utf-8'))
print('Recorded as of:', recorded['as_of'])
print('Scope:', recorded['scope'])

Recorded as of: 2026-10-22
Scope: offline_observational_no_strategy_changes


## 1. Sample independence

Original read-only SQLite extraction (the exact query is also in the reproduction script):
```sql
SELECT * FROM (
 SELECT * FROM recommendation_outcomes
 WHERE quality_status='valid' AND pick_date<=?
 ORDER BY pick_date DESC, ticker LIMIT 1000
) ORDER BY pick_date, ticker DESC;
```
The learner then requires a non-null 20-day outcome and a valid objective. Balanced-day concentration and non-overlapping windows are diagnostics, not estimates of a true independent sample size. The short-history block intervals are exploratory.

In [2]:
sample, daily = sample_audit(rows, calendar)
assert sample['rows'] == recorded['sample']['rows']
assert sample['block_bootstrap'] == recorded['sample']['block_bootstrap']
for name in ('rows', 'pick_days', 'entry_days', 'tickers', 'balanced_day_equivalent', 'overlap', 'fold_boundaries', 'block_bootstrap'):
    print(name, json.dumps(sample[name], ensure_ascii=False))

rows 310
pick_days 35
entry_days 35
tickers 243
balanced_day_equivalent 18.002997377294868
overlap {"10": {"mapped_windows": 310, "same_ticker_overlapping_rows": 44, "same_ticker_nonoverlap_rows": 268, "nonoverlapping_date_windows": 4}, "20": {"mapped_windows": 310, "same_ticker_overlapping_rows": 62, "same_ticker_nonoverlap_rows": 251, "nonoverlapping_date_windows": 2}}
fold_boundaries [{"fold": 1, "training_rows": 130, "validation_rows": 60, "training_last_pick": "2026-09-04", "validation_first_pick": "2026-09-04", "same_pick_day_boundary": true, "training_targets_not_finished_before_validation": 57}, {"fold": 2, "training_rows": 190, "validation_rows": 60, "training_last_pick": "2026-09-09", "validation_first_pick": "2026-09-09", "same_pick_day_boundary": true, "training_targets_not_finished_before_validation": 106}, {"fold": 3, "training_rows": 250, "validation_rows": 60, "training_last_pick": "2026-09-15", "validation_first_pick": "2026-09-15", "same_pick_day_boundary": true, "tra

## 2. Result lifetime

The registry freezes the report/driver bytes observed now, not their historical bytes. A complete engine/data dependency lock and a new out-of-sample result series are still needed. Korean price-file coverage is not US coverage. Missing prospective results mean pending, not an expired effect.

In [3]:
life = recorded['lifetime']
print('Price files:', life['price_files'], 'latest stored date:', life['latest_price_day'])
for study in life['studies']:
    print(study['study'], study['status'], 'hash recorded:', study['hash_frozen_on'])

Price files: 2971 latest stored date: 2026-09-28
low_vol await_frozen_forward_results hash recorded: 2026-10-22
high52w await_frozen_forward_results hash recorded: 2026-10-22
reversal_1w await_frozen_forward_results hash recorded: 2026-10-22
high_vol_exclusion await_frozen_forward_results hash recorded: 2026-10-22
bear_ma20_reentry await_frozen_forward_results hash recorded: 2026-10-22
holding_period await_frozen_forward_results hash recorded: 2026-10-22
us_baseline await_frozen_forward_results hash recorded: 2026-10-22
exit_variants await_frozen_forward_results hash recorded: 2026-10-22


## 3. Source consistency

Original query: `SELECT quality_id, created_at, ticker, price_date, source, reference_source, close, reference_close, status, reason FROM price_quality WHERE substr(created_at,1,10)<=? ORDER BY quality_id`.

Retain the last observation by day/ticker/source/reference label/session. Source labels and observation timestamps do not establish quote timestamps or actual fallback provider. These differences cannot be called provider error rates. Rows without an independent reference remain in the denominator for coverage, not in the price-difference denominator.

In [4]:
with (folder / 'price_pairs.csv').open(encoding='utf-8-sig', newline='') as file:
    pairs = list(csv.DictReader(file))
paired = [r for r in pairs if r['abs_difference_pct'] != '']
assert len(pairs) == recorded['prices']['deduplicated_observations']
print('Raw rows:', recorded['prices']['raw']['rows'])
print('Latest observations:', len(pairs), 'paired:', len(paired))
print('Strict simultaneous comparable pairs verified:', recorded['prices']['strict_comparable_pairs'])
for group in recorded['prices']['groups']:
    if group['paired']:
        print(json.dumps(group, ensure_ascii=False))

Raw rows: 654572
Latest observations: 13122 paired: 5438
Strict simultaneous comparable pairs verified: 0
{"source": "naver", "reference_label": "pykrx", "session": "after_1600", "latest_observations": 10, "paired": 10, "same_observation_date": 10, "median_abs_difference_pct": 0.0, "p95_abs_difference_pct": 0.0, "over_0_5pct": 0, "over_0_5pct_rate": 0.0}
{"source": "naver", "reference_label": "toss", "session": "after_1600", "latest_observations": 205, "paired": 205, "same_observation_date": 196, "median_abs_difference_pct": 0.0, "p95_abs_difference_pct": 0.19323671497584216, "over_0_5pct": 9, "over_0_5pct_rate": 0.04390243902439024}
{"source": "naver", "reference_label": "toss", "session": "early_close", "latest_observations": 189, "paired": 189, "same_observation_date": 188, "median_abs_difference_pct": 0.0, "p95_abs_difference_pct": 0.0, "over_0_5pct": 1, "over_0_5pct_rate": 0.005291005291005291}
{"source": "naver", "reference_label": "toss", "session": "intraday", "latest_observati